# RetinaGuard Colab MVP

Upload one permitted colour fundus image, run the image-quality gate, perform five-grade diabetic-retinopathy inference, apply the confidence gate and deterministic referral rules, then generate and validate a constrained OpenRouter referral draft. Unsafe LLM output is replaced by a deterministic fallback. This research prototype is not a diagnostic device and every output requires licensed clinician review.

In [ ]:
!pip install -q -r https://raw.githubusercontent.com/MauriceChang1027/PE6201-End-of-Course-Project/main/requirements-colab.txt

import os
import subprocess
import sys
from pathlib import Path

repo_path = Path('/content/PE6201-End-of-Course-Project')
if repo_path.exists():
    subprocess.run(['git', '-C', str(repo_path), 'pull', '--ff-only'], check=True)
else:
    subprocess.run([
        'git', 'clone',
        'https://github.com/MauriceChang1027/PE6201-End-of-Course-Project.git',
        str(repo_path),
    ], check=True)
sys.path.insert(0, str(repo_path))
os.environ.setdefault('KERAS_BACKEND', 'tensorflow')
print('RetinaGuard setup complete.')

In [ ]:
from google.colab import files
from PIL import Image
import matplotlib.pyplot as plt

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload exactly one JPG or PNG fundus image.')
image_path = next(iter(uploaded))
image = Image.open(image_path).convert('RGB')
plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.axis('off')
plt.show()

In [ ]:
from retinaguard import RetinaGuardPipeline
from retinaguard.config import GRADE_LABELS

analysis = RetinaGuardPipeline().analyse(image)
triage = analysis.triage

print(f'Quality gate passed: {analysis.quality.passed}')
for issue in analysis.quality.issues:
    print(f'  Rejection reason: {issue.message}')
print('Quality measurements:', analysis.quality.metrics)

if triage is not None:
    prediction = triage.prediction
    print(f'\nGrade: {prediction.grade} - {prediction.label}')
    print(f'Confidence: {prediction.confidence:.1%}')
    print(f'Urgency: {triage.urgency}')
    print(f'Action: {triage.action}')
    print(f'Abstained: {triage.abstained}')
    print('\nClass probabilities')
    for grade, (label, probability) in enumerate(zip(GRADE_LABELS, prediction.probabilities)):
        print(f'  Grade {grade} - {label}: {probability:.1%}')
else:
    print('Vision-model inference was not run. Upload a clearer fundus image.')

In [ ]:
from getpass import getpass
from retinaguard import OpenRouterClient

if triage is None:
    print('The image failed the quality gate. Referral generation is disabled.')
elif triage.abstained:
    print('The system abstained. Do not generate a normal referral conclusion; arrange human review.')
else:
    openrouter_api_key = getpass('OpenRouter API key: ')
    patient_reference = input('Demo patient reference [DEMO-001]: ').strip() or 'DEMO-001'
    client = OpenRouterClient(openrouter_api_key)
    referral = client.generate_safe_referral(triage, patient_reference)
    print('\nReferral draft for clinician review only\n')
    print(referral.text)
    print(f'\nLLM validation passed: {referral.validation_passed}')
    if referral.used_fallback:
        print('The LLM output was rejected; a deterministic fallback is shown.')
        for issue in referral.validation_issues:
            print(f'  Validation issue: {issue}')